# Instrument `G_t^arg -> G_{t+1}^arg` — graphes d'argumentation datés

Issue #13310. Substrat réutilisé depuis :

- `Argument_Analysis_Ontology_AIF.ipynb` — schéma AIF (nœuds I-nodes/S-nodes, arêtes).
- `Argument_Analysis_Dung_AF_Semantics.ipynb` — sémantique de Dung (admissible, preferred, grounded, ideal).
- `Argument_Analysis_Ranking_Semantics.ipynb` — sémantique graduée (h-categorizer).

## Pourquoi cet instrument

Comparer deux graphes d'argumentation sur deux dates est facile : `d(G_t, G_{t+1}) > 0`. C'est aussi inutile sans plancher : deux graphes tirés d'**une même** période diffèrent par le seul bruit d'échantillonnage. Ce notebook livre l'instrument qui rend la comparaison interprétable — un graphe AIF-conforme, deux mesures d'écart de familles différentes, un contrôle négatif par découpage d'une même période, un contrôle positif par mutation connue.

## 1. Substrat minimal — redéfinition locale

Les fonctions qui suivent sont reprises de `Argument_Analysis_Dung_AF_Semantics.ipynb` (cellules `is_conflict_free`, `defeats`, `is_admissible`, `ideal`, `powerset`) — non pour les redéfinir, mais pour rendre ce notebook **autonome** : il n'importe pas le notebook source. Les fonctions référencent le même vocabulaire Dung (`af = (args, attacks)`, `attacks ⊆ args×args`) et n'ont aucune dépendance externe.

In [1]:
from itertools import chain, combinations
from typing import Iterable, Tuple, FrozenSet, List, Dict, Set

Attack = Tuple[str, str]  # (attacker, target)
AF = Tuple[FrozenSet[str], FrozenSet[Attack]]

MAX_ENUM_ARGS = 18  # Au-delà : labeling grounded polynomiale (cf cellule grounded du substrat Dung).

def powerset(iterable: Iterable) -> List[Tuple]:
    """Toutes les sous-parties de `iterable` (cf. Dung notebook, cellule powerset)."""
    s = list(iterable)
    return list(chain.from_iterable(combinations(s, r) for r in range(len(s) + 1)))

def defeats(af: AF, S: Iterable[str]) -> Set[str]:
    args, attacks = af
    S = set(S)
    return {b for (a, b) in attacks if a in S}

def is_conflict_free(af: AF, S: Iterable[str]) -> bool:
    args, attacks = af
    S = set(S)
    return not any((b in S) for (a, b) in attacks if a in S)

def is_admissible(af: AF, S: Iterable[str]) -> bool:
    args, attacks = af
    S = set(S)
    return is_conflict_free(af, S) and set(defeats(af, S)) <= {b for (a, b) in attacks if a in S} - S

def grounded_labeling(af: AF) -> Dict[str, str]:
    """Labeling grounded (in, out, undec) par point fixe.
    Convention Dung : tout argument non défaite par un 'in' est 'in' ou 'undec'.
    On itère jusqu'au point fixe depuis (undec, undec, ...).
    """
    args, attacks = af
    in_set, out_set, undec_set = set(), set(), set(args)
    changed = True
    while changed:
        changed = False
        # Tout argument battu par un 'in' devient 'out'
        new_out = {b for (a, b) in attacks if a in in_set}
        if new_out - out_set:
            out_set |= new_out - out_set
            undec_set -= new_out
            changed = True
        # Tout argument dont tous les defeaters sont 'out' devient 'in'
        defeat_by = {b: [a for (a, bb) in attacks if bb == b] for (a, b) in attacks}
        for b in list(undec_set):
            defeaters = defeat_by.get(b, [])
            if defeaters and all(d in out_set for d in defeaters):
                in_set.add(b)
                undec_set.discard(b)
                changed = True
    return {a: ("in" if a in in_set else "out" if a in out_set else "undec") for a in args}

def preferred_extensions(af: AF) -> List[FrozenSet[str]]:
    """Extensions preferred = ensembles ⊆-maximaux admissibles.
    Implémentation naïve énumérative plafonnee a MAX_ENUM_ARGS.
    Au-dela, retombe sur le labeling grounded (labeling unique, pas d'extensions).
    """
    args, _ = af
    if len(args) > MAX_ENUM_ARGS:
        # Substitut : on retourne le set des arguments 'in' du grounding comme proxy
        # (ce n'est PAS preferred, mais c'est polynomial). Le label le dit explicitement.
        lbl = grounded_labeling(af)
        return [frozenset(a for a, v in lbl.items() if v == "in")]
    candidates = [set(S) for S in powerset(args) if is_admissible(af, S)]
    maximal = []
    for S in candidates:
        if not any(S < T for T in candidates):
            maximal.append(frozenset(S))
    return sorted(maximal, key=lambda s: sorted(s))

print(f"Substrat OK : powerset, defeats, is_conflict_free, is_admissible, preferred_extensions (MAX_ENUM_ARGS={MAX_ENUM_ARGS} -> grounded labeling au-dela).")

Substrat OK : powerset, defeats, is_conflict_free, is_admissible, preferred_extensions (MAX_ENUM_ARGS=18 -> grounded labeling au-dela).


## 2. Construction `G_t^arg` depuis un corpus daté

Un corpus daté est une liste d'événements argumentatifs :

- `(date, "énoncé", attaquant_ou_none)` — un argument posé à `date`, qui attaque optionnellement un argument préexistant.

**Critère d'inclusion d'un nœud** (acceptance 1) : un énoncé devient argument `A_k` si `k ≤ N` où `N` est le plafond global, ET s'il porte une attaque vers un argument déjà présent dans la fenêtre de la période (sinon, c'est un énoncé flottant qui n'attaque personne et n'est attaqué par personne — il dégrade la mesure d'écart, on l'inclut quand même car l'absence de structure est une information).
**Fenêtrage** : `periods = sorted({date for ...})` ; `G_t^arg` agrège tous les arguments datés `≤ periodes[t]` ET `> periodes[t-1]`.

In [2]:
from collections import defaultdict
import random

ArgumentEvent = Tuple[str, str, str]  # (date, text, attacker_or_"")

def build_af_from_corpus(events: List[ArgumentEvent]) -> Tuple[AF, Dict[str, str]]:
    """Construit un AF à partir d'événements datés.
    Renvoie (af, text_map) où text_map[arg_id] = texte de l'énoncé.
    Chaque énoncé devient un nœud A_<i>. Si l'énoncé attaque un argument préexistant, on crée une arête."""
    args = []
    attacks = []
    text_map = {}
    for i, (date, text, attacker) in enumerate(events):
        arg_id = f"A_{i:03d}"
        args.append(arg_id)
        text_map[arg_id] = text
        if attacker and attacker in text_map:  # attacker réfère à un arg_id
            attacks.append((arg_id, attacker))
    return (frozenset(args), frozenset(attacks)), text_map

def filter_period(events: List[ArgumentEvent], period: str, prev_period: str = "") -> List[ArgumentEvent]:
    """Filtre les événements de la période (date == period) ET les attaques
    qui pointent vers un argument de la période précédente (continuité)."""
    return [e for e in events if e[0] == period]

def cumulative_af_until(events: List[ArgumentEvent], period: str) -> Tuple[AF, Dict[str, str]]:
    """AF cumulatif : tous les arguments datés ≤ period."""
    cum = [e for e in events if e[0] <= period]
    return build_af_from_corpus(cum)

print("Constructeurs OK : build_af_from_corpus, filter_period, cumulative_af_until.")

Constructeurs OK : build_af_from_corpus, filter_period, cumulative_af_until.


## 3. Deux mesures d'écart (critère 2)

- **Structurelle** : `Jaccard(args_A, args_B)` + `Jaccard(attacks_A, attacks_B)` — symétrique, bornée [0,1]. Indépendante de la sémantique Dung.
- **Sémantique** : `Jaccard(preferred_extensions(A), preferred_extensions(B))` — variation sur le **résultat** du calcul d'acceptabilité. Deux graphes avec mêmes nœuds/arcs mais attaques inversées peuvent avoir mêmes nœuds/arcs mais extensions preferred différentes.

Distance de Jaccard : `d(X, Y) = 1 - |X ∩ Y| / |X ∪ Y|` (0 = identique, 1 = disjoint).

In [3]:
def jaccard_distance(X: Iterable, Y: Iterable) -> float:
    """Distance de Jaccard bornée [0, 1]."""
    X, Y = set(X), set(Y)
    union = X | Y
    if not union:
        return 0.0
    return 1.0 - len(X & Y) / len(union)

def structural_distance(af_a: AF, af_b: AF) -> Dict[str, float]:
    """Mesure structurelle : Jaccard sur args et attacks séparément."""
    args_a, att_a = af_a
    args_b, att_b = af_b
    return {
        "d_args": jaccard_distance(args_a, args_b),
        "d_attacks": jaccard_distance(att_a, att_b),
        "d_args_size_a": len(args_a),
        "d_args_size_b": len(args_b),
    }

def semantic_distance(af_a: AF, af_b: AF) -> Dict[str, float]:
    """Mesure sémantique : Jaccard sur les ensembles d'extensions preferred."""
    ext_a = set(preferred_extensions(af_a))
    ext_b = set(preferred_extensions(af_b))
    return {
        "d_preferred": jaccard_distance(ext_a, ext_b),
        "n_ext_a": len(ext_a),
        "n_ext_b": len(ext_b),
    }

def both_distances(af_a: AF, af_b: AF) -> Dict[str, float]:
    """Renvoie les deux mesures fusionnées."""
    out = {"family_a_vs_b": True}
    out.update(structural_distance(af_a, af_b))
    out.update(semantic_distance(af_a, af_b))
    return out

print("Mesures OK : structural_distance, semantic_distance, both_distances.")

Mesures OK : structural_distance, semantic_distance, both_distances.


## 4. Corpus synthétique de démonstration

On construit un corpus **synthétique** (critère 4) où le changement entre dates est connu par construction. Trois périodes : `T0`, `T1`, `T2`. À `T0`, 15 arguments en réseau clairsemé. À `T1`, on ajoute 8 arguments dont 3 attaquent des `T0`. À `T2`, on ajoute 4 arguments (au lieu d'un retrait — voir discussion §9 : l'hypothèse monotone).

**Ordre de grandeur attendu (avant exécution, critère 4)** :

- `d_args(T0, T1)` ≈ 8 / 23 ≈ **0.35** (8 ajoutés sur 23 total).
- `d_attacks(T0, T1)` ≈ variable, ~3 attaques sur total petit.
- `d_args(T1, T2)` ≈ 4 / 27 ≈ **0.15** (4 ajoutés sur 27).
- `d_preferred` ≈ variable, 0 si extensions stables, >0 sinon.

Note : G_T2 dépasse `MAX_ENUM_ARGS=18` donc `preferred_extensions` bascule vers `grounded_labeling` (labeling unique, polynomial). C'est un **proxy** assumé — voir §9.

In [4]:
import random
random.seed(42)  # Reproductibilité

# Tailles choisies pour rester sous MAX_ENUM_ARGS=18 dans preferred_extensions.
# G_T2 cumulatif atteindra 15+8+4 = 27 args -> MAX_ENUM_ARGS bascule vers grounded labeling.

def make_synthetic_corpus() -> List[ArgumentEvent]:
    """Corpus synthétique à 3 périodes avec changement connu par construction.
    Tailles plafonnées pour rester sous MAX_ENUM_ARGS=18."""
    events = []
    # T0 : 15 arguments, attaques clairsemées (densité ~0.10)
    arg_ids_t0 = [f"A_{i:03d}" for i in range(15)]
    for i, aid in enumerate(arg_ids_t0):
        target = arg_ids_t0[i - 1] if i > 0 and random.random() < 0.20 else ""
        events.append(("T0", f"énoncé T0 {i}", target))
    # T1 : 8 nouveaux arguments, dont 3 attaquent des T0
    arg_ids_t1 = [f"A_{i:03d}" for i in range(15, 23)]
    for j, aid in enumerate(arg_ids_t1):
        if j < 3:
            target = arg_ids_t0[random.randint(0, 14)]
        else:
            target = ""
        events.append(("T1", f"énoncé T1 {j}", target))
    # T2 : 4 nouveaux arguments (au lieu d'un retrait — voir discussion §9)
    arg_ids_t2 = [f"A_{i:03d}" for i in range(23, 27)]
    for k, aid in enumerate(arg_ids_t2):
        target = arg_ids_t1[random.randint(0, 7)] if random.random() < 0.50 else ""
        events.append(("T2", f"énoncé T2 {k}", target))
    return events

corpus_t0_t1_t2 = make_synthetic_corpus()
print(f"Corpus synthétique : {len(corpus_t0_t1_t2)} événements")
print(f"  T0 : {sum(1 for e in corpus_t0_t1_t2 if e[0] == 'T0')} arguments")
print(f"  T1 : {sum(1 for e in corpus_t0_t1_t2 if e[0] == 'T1')} arguments")
print(f"  T2 : {sum(1 for e in corpus_t0_t1_t2 if e[0] == 'T2')} arguments")
print(f"  Total cumul G_T2 : {sum(1 for e in corpus_t0_t1_t2)} arguments (plafond {MAX_ENUM_ARGS} -> grounded au-delà)")

Corpus synthétique : 27 événements
  T0 : 15 arguments
  T1 : 8 arguments
  T2 : 4 arguments
  Total cumul G_T2 : 27 arguments (plafond 18 -> grounded au-delà)


## 5. Construction des graphes cumulatifs

On est dans une hypothèse **monotone** : `G_T0 ⊆ G_T1 ⊆ G_T2` au sens ensembliste (les arguments ne sont pas retirés — voir discussion §6). Le retrait est un chantier ultérieur.

In [5]:
af_t0, text_t0 = cumulative_af_until(corpus_t0_t1_t2, "T0")
af_t1, text_t1 = cumulative_af_until(corpus_t0_t1_t2, "T1")
af_t2, text_t2 = cumulative_af_until(corpus_t0_t1_t2, "T2")

print(f"G_T0 : {len(af_t0[0])} args, {len(af_t0[1])} attacks")
print(f"G_T1 : {len(af_t1[0])} args, {len(af_t1[1])} attacks")
print(f"G_T2 : {len(af_t2[0])} args, {len(af_t2[1])} attacks")

G_T0 : 15 args, 5 attacks
G_T1 : 23 args, 8 attacks
G_T2 : 27 args, 10 attacks


## 6. Contrôle positif (critère 4) — changement connu

On compare `G_T0` à `G_T1` : on **sait** que 8 arguments ont été ajoutés et 3 attaques inter-périodes. **Avant exécution** l'ordre de grandeur attendu est :

- `d_args ≈ 8/23 = 0.35`,
- `d_attacks ≈ 3 / N_total_attacks` (variable, dépend des attaques intra-période).
- `d_preferred` ≥ 0 ; variation dépendante de la structure.

Si la mesure détecte un changement de cet ordre de grandeur, l'instrument fonctionne.</cell id>

In [6]:
d_t0_t1 = both_distances(af_t0, af_t1)
d_t1_t2 = both_distances(af_t1, af_t2)
d_t0_t2 = both_distances(af_t0, af_t2)

print("=== Contrôle positif : changement connu par construction ===")
print(f"G_T0 vs G_T1 (10 args ajoutés, ~4 attaques) :")
print(f"  d_args      = {d_t0_t1['d_args']:.4f}  (attendu ≈ 0.33)")
print(f"  d_attacks   = {d_t0_t1['d_attacks']:.4f}  (attendu ≈ 0.50)")
print(f"  d_preferred = {d_t0_t1['d_preferred']:.4f}")
print(f"  n_ext_a={d_t0_t1['n_ext_a']}, n_ext_b={d_t0_t1['n_ext_b']}")
print()
print(f"G_T1 vs G_T2 (5 args ajoutés) :")
print(f"  d_args      = {d_t1_t2['d_args']:.4f}  (attendu ≈ 0.14 = 5/35)")
print(f"  d_attacks   = {d_t1_t2['d_attacks']:.4f}")
print(f"  d_preferred = {d_t1_t2['d_preferred']:.4f}")
print()
print(f"G_T0 vs G_T2 (15 args ajoutés cumulés) :")
print(f"  d_args      = {d_t0_t2['d_args']:.4f}  (attendu ≈ 0.43 = 15/35)")
print(f"  d_attacks   = {d_t0_t2['d_attacks']:.4f}")
print(f"  d_preferred = {d_t0_t2['d_preferred']:.4f}")

=== Contrôle positif : changement connu par construction ===
G_T0 vs G_T1 (10 args ajoutés, ~4 attaques) :
  d_args      = 0.3478  (attendu ≈ 0.33)
  d_attacks   = 0.3750  (attendu ≈ 0.50)
  d_preferred = 1.0000
  n_ext_a=16, n_ext_b=1

G_T1 vs G_T2 (5 args ajoutés) :
  d_args      = 0.1481  (attendu ≈ 0.14 = 5/35)
  d_attacks   = 0.2000
  d_preferred = 0.0000

G_T0 vs G_T2 (15 args ajoutés cumulés) :
  d_args      = 0.4444  (attendu ≈ 0.43 = 15/35)
  d_attacks   = 0.5000
  d_preferred = 1.0000


## 7. Contrôle négatif (critère 3) — plancher de bruit

On découpe `T1` en deux moitiés arbitraires (pair/impair sur l'ordre d'arrivée). On construit `G_a` (moitié pair) et `G_b` (moitié impair), puis on mesure. **Cette distance est le plancher de bruit** : tout écart inter-dates qui ne le dépasse pas n'est pas un changement.

**Le plancher est publié avec toute mesure d'écart.**

In [7]:
def split_period_in_half(events: List[ArgumentEvent], period: str) -> Tuple[List[ArgumentEvent], List[ArgumentEvent]]:
    """Découpe les événements d'une période en pair/impair."""
    sub = [e for e in events if e[0] == period]
    half = len(sub) // 2
    return sub[::2], sub[1::2]

def cumulative_af_from_subset(events_subset: List[ArgumentEvent], period: str) -> Tuple[AF, Dict[str, str]]:
    """Construit l'AF cumulatif jusqu'à period en n'utilisant que les événements de `events_subset`
    qui sont antérieurs OU ÉGAUX à period."""
    # On prend TOUT ce qui vient avant period, plus la moitié retenue de period.
    cum = [e for e in events_subset if e[0] <= period]
    return build_af_from_corpus(cum)

# On prend la période T1 (10 arguments) qu'on split en 2 moitiés
sub_a, sub_b = split_period_in_half(corpus_t0_t1_t2, "T1")
print(f"T1 moitié pair : {len(sub_a)} arguments")
print(f"T1 moitié impair : {len(sub_b)} arguments")

# Mais le split ne touche que T1 : G_a et G_b diffèrent seulement par leur moitié T1,
# et incluent tous les arguments T0 (communs). On construit donc les AF cumulatifs
# qui incluent T0 + moitié de T1, en rebranchant les attaques vers T0 correctement.
def build_noise_floor(events: List[ArgumentEvent], period: str) -> Dict[str, float]:
    """Construit le plancher de bruit pour une période donnée par split pair/impair."""
    sub_a, sub_b = split_period_in_half(events, period)
    # G_a : cumulatif jusqu'à period avec sub_a uniquement (pour cette période)
    # Approximation : on re-trie les événements en mettant sub_a à la place de T1
    events_a = [e for e in events if e[0] < period] + sub_a
    events_b = [e for e in events if e[0] < period] + sub_b
    af_a, _ = build_af_from_corpus(events_a)
    af_b, _ = build_af_from_corpus(events_b)
    return both_distances(af_a, af_b)

noise_t1 = build_noise_floor(corpus_t0_t1_t2, "T1")
print("\n=== Contrôle négatif : plancher de bruit sur T1 (split pair/impair) ===")
print(f"  d_args_noise      = {noise_t1['d_args']:.4f}")
print(f"  d_attacks_noise   = {noise_t1['d_attacks']:.4f}")
print(f"  d_preferred_noise = {noise_t1['d_preferred']:.4f}")

T1 moitié pair : 4 arguments
T1 moitié impair : 4 arguments

=== Contrôle négatif : plancher de bruit sur T1 (split pair/impair) ===
  d_args_noise      = 0.0000
  d_attacks_noise   = 0.3750
  d_preferred_noise = 0.0000


## 8. Synthèse — mesures inter-dates vs plancher

Tableau récapitulatif : pour chaque écart inter-dates, on publie la valeur **et** le plancher de bruit correspondant. Un écart qui ne dépasse pas son plancher n'est pas un changement — c'est l'instrument qui respire.

In [8]:
print("=" * 70)
print("Mesure                 | Valeur   | Plancher | Verdict")
print("-" * 70)

rows = [
    ("d_args(G_T0, G_T1)", d_t0_t1['d_args'], noise_t1['d_args']),
    ("d_attacks(G_T0, G_T1)", d_t0_t1['d_attacks'], noise_t1['d_attacks']),
    ("d_preferred(G_T0, G_T1)", d_t0_t1['d_preferred'], noise_t1['d_preferred']),
    ("d_args(G_T1, G_T2)", d_t1_t2['d_args'], noise_t1['d_args']),
    ("d_attacks(G_T1, G_T2)", d_t1_t2['d_attacks'], noise_t1['d_attacks']),
    ("d_preferred(G_T1, G_T2)", d_t1_t2['d_preferred'], noise_t1['d_preferred']),
    ("d_args(G_T0, G_T2)", d_t0_t2['d_args'], noise_t1['d_args']),
    ("d_attacks(G_T0, G_T2)", d_t0_t2['d_attacks'], noise_t1['d_attacks']),
    ("d_preferred(G_T0, G_T2)", d_t0_t2['d_preferred'], noise_t1['d_preferred']),
]

for name, val, floor in rows:
    verdict = "✓ signal" if val > floor + 0.05 else "≈ bruit"
    print(f"{name:30s} | {val:.4f}   | {floor:.4f}  | {verdict}")

Mesure                 | Valeur   | Plancher | Verdict
----------------------------------------------------------------------
d_args(G_T0, G_T1)             | 0.3478   | 0.0000  | ✓ signal
d_attacks(G_T0, G_T1)          | 0.3750   | 0.3750  | ≈ bruit
d_preferred(G_T0, G_T1)        | 1.0000   | 0.0000  | ✓ signal
d_args(G_T1, G_T2)             | 0.1481   | 0.0000  | ✓ signal
d_attacks(G_T1, G_T2)          | 0.2000   | 0.3750  | ≈ bruit
d_preferred(G_T1, G_T2)        | 0.0000   | 0.0000  | ≈ bruit
d_args(G_T0, G_T2)             | 0.4444   | 0.0000  | ✓ signal
d_attacks(G_T0, G_T2)          | 0.5000   | 0.3750  | ✓ signal
d_preferred(G_T0, G_T2)        | 1.0000   | 0.0000  | ✓ signal


## 9. Discussion — limites et étapes suivantes

### Hypothèse monotone

L'instrument suppose que `G_T0 ⊆ G_T1 ⊆ G_T2`. Un graphe d'argumentation **non monotone** (arguments retirés parce que réfutés) demanderait une reconstruction par fenêtre glissante plutôt que cumulatif. Cela reste à faire.

### `d_preferred` et complexité

`preferred_extensions` est implémenté en énumérant tous les sous-ensembles : exponentiel en `|args|`. Pour des graphes > 25 arguments, préférer une approche par *labeling* (cf. cellule `grounded` du notebook Dung) qui est polynomiale.

### Branchements

- **#13303** — chaîne parente (catégories lexicales → catégories argumentatives → graphes).
- **#13309** — protocole de choix de corpus daté réel (à appliquer quand ce notebook est validé sur synthétique).

### Ce que ce notebook NE FAIT PAS

- Aucun corpus réel. La validation est entièrement synthétique + contrôle négatif.
- Aucune conclusion sur les formes relationnelles (humour, désir, attachement) — l'instrument est neutre.